In [ ]:
# ============================================================
# ENGINEERING MEASUREMENT UNCERTAINTIES - GUM CALC  
# ============================================================

import sys

sys.path.insert(0, r"C:\Users\victo\Desktop\Python\gum-calc")

try:
    from gum_calc import (
        uncertainty_type_A,
        uncertainty_type_B_from_resolution,
        uncertainty_type_B_uniform,
        uncertainty_type_B_relative,
        uncertainty_type_exact,
        full_gum_analysis,
        generate_bilan,
        generate_bilan_regression,
        generate_annexe,
        full_pipeline_regression_to_measurand,
    )
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "gum_calc introuvable : vérifiez que le chemin passé à "
        "sys.path.insert ci-dessus pointe bien vers le dossier qui "
        "contient gum_calc.py sur cette machine."
    ) from e

# UncertaintyInput est une dataclass — accès par attribut UNIQUEMENT :
#   u_X.u    → incertitude-type
#   u_X.nu   → degrés de liberté
#   u_X.N    → nombre de mesures (type A seulement)
#   u_X.s    → écart-type empirique (type A seulement)
#   u_X.mean → moyenne empirique (type A seulement)
# NE PAS écrire u_X["u"] ou u_X["mean"] → TypeError garanti

print("gum_calc chargé avec succès.")

In [ ]:
# ============================================================
# MESURANDE — Exemple : Résistance (R), à dupliquer/adapter pour
# chaque grandeur du TP (en renommant systématiquement le suffixe
# _R par le suffixe de la grandeur traitée dans la cellule copiée).
# ============================================================

# --- Sources d'incertitude : NE GARDER QU'UNE SEULE ligne par variable,
# selon la nature de la mesure ; les quatre autres restent en commentaire.

# Type A : nommer la liste pour pouvoir en extraire la moyenne via .mean
U_values = [5.02, 5.01, 5.03, 5.02, 5.00]   # N = 5 mesures répétées
u_U      = uncertainty_type_A(U_values)       # u_U.mean est synchronisé avec la liste
# u_U = uncertainty_type_B_from_resolution(resolution=0.01)   # Type B : résolution instrument
# u_U = uncertainty_type_B_uniform(half_width=0.01)           # Type B : demi-largeur connue
# u_U = uncertainty_type_B_relative(u_standard=0.01)          # Type B : u connue (notice)
# u_U = uncertainty_type_exact()                               # Constante exacte

u_I = uncertainty_type_B_from_resolution(resolution=0.001)   # Type B : résolution instrument
# u_I = uncertainty_type_A([...])
# u_I = uncertainty_type_B_uniform(half_width=...)
# u_I = uncertainty_type_B_relative(u_standard=...)
# u_I = uncertainty_type_exact()

# --- Valeurs nominales ---
# Règle : si type A, utiliser .mean (synchronisé) — jamais un littéral codé en dur.
#         si type B, la valeur nominale est fournie par la mesure directe (OK codée en dur).
nominales_R = {
    "U": u_U.mean,   # ← toujours synchronisé avec U_values
    "I": 0.502,      # ← type B : valeur nominale externe, OK
}

# --- Incertitudes ---
incertitudes_R = {
    "U": u_U,
    "I": u_I,
}

# --- Analyse GUM ---
res_R = full_gum_analysis(
    formula_str        = "U / I",
    variable_names     = ["U", "I"],
    nominal_values     = nominales_R,
    uncertainty_inputs = incertitudes_R,
)

# --- Bilan LaTeX : _res_precomputed réutilise le calcul ci-dessus
# plutôt que de relancer toute l'analyse GUM une seconde fois ---
bilan_R = generate_bilan(
    measurand_name     = "Résistance",
    measurand_symbol   = "R",
    formula_str        = "U / I",
    variable_names     = ["U", "I"],
    variable_symbols   = {"U": "U", "I": "I"},
    variable_units     = {"U": r"\volt", "I": r"\ampere"},
    nominal_values     = nominales_R,
    uncertainty_inputs = incertitudes_R,
    measurand_unit     = r"\ohm",
    _res_precomputed   = res_R,
)

# --- Affichage console ---
print(f"R = {res_R['result_rounded']} ± {res_R['U_rounded']}")
print(f"uc = {res_R['uc']:.4g}  |  k = {res_R['k']:.3f}  |  ν_eff = {res_R['nu_eff']:.1f}")
print("Budget :", {k: f"{v:.1f}%" for k, v in res_R['budget'].items()})

In [ ]:
# ============================================================
# GÉNÉRATION DU LATEX
# ============================================================

print(generate_annexe([
    bilan_R,
    # bilan_<G2>,  # ajouter ici un bilan_<G> par mesurande de la cellule précédente, dans l'ordre du CR
]))